# news_macro_hot_issue — 시간대를 정하면 거시 핫이슈 3줄 정리

`news_extract`·`news_track`이 쌓은 `llm_source.jsonl`에서 **정한 시간대의 거시 기사**만 골라, **소식 매체 수(`news_outlets`)가 높은 사건 3개**를 OpenAI로 한 줄씩 정리한다.

```
news_extract  ─ llm_source.jsonl에 기사 한 줄(article_type: 거시 유형만, 한국어)
news_track    ─ 같은 줄에 story_id(사건 번호), news_outlets(소식 매체 수)를 채움
   ▼
news_macro_hot_issue_tmp.ipynb ─ 시간대 거르기 → 거시·사건 기사만 → 사건마다 하나 → 매체 수 상위 3개 → OpenAI 3줄
   ▼  macro_hot_issues.jsonl
- ~
- ~
- ~
```

- 이 노트북은 `llm_source.jsonl`을 **읽기만** 한다.
- 프롬프트: `prompt/macro_hot_issue_prompt.md`(proj/prompt에서 복제).

## 1. 설정

- `START_KST`, `END_KST`: 정리할 시간대(한국 시각, `START_KST` 이상 `END_KST` 미만).
- `TIME_COLUMN`: 시간대를 어느 시각으로 거를지. `published_utc`(기사 발행 시각) 또는 `first_seen_utc`(우리가 처음 수집한 시각).
- `TOP_N`: 고를 사건 수(3줄이면 3). `MAX_ARTICLES_PER_ISSUE`: 사건마다 LLM에 넘길 기사 수(대표 기사 포함).
- `RUN_LLM = False`면 고른 사건과 LLM에 넘길 입력만 보여 주고 **OpenAI를 부르지 않는다.**
- 키는 proj 폴더 `.env`의 `OPENAI_API_KEY`이며 값은 출력하지 않는다.

In [ ]:
import datetime as dt
import json
import urllib.error
import urllib.request
from pathlib import Path

import pandas as pd
from IPython.display import display

# ===== 여기만 바꿔서 쓴다 =====
ROOT = Path.cwd()                      # news_search 폴더(이 노트북이 있는 폴더에서 연다)
PROJ = ROOT.parents[2]                 # proj 폴더(.env가 있는 곳)
RESULT_DIR = ROOT / 'source' / 'live'  # news_extract·news_track이 함께 쓰는 결과 폴더
HOT_PROMPT_PATH = ROOT / 'prompt' / 'macro_hot_issue_prompt.md'   # proj/prompt에서 복제
LLM_MODEL = 'gpt-6.1-terra'
RUN_LLM = False          # True여야 OpenAI로 3줄 정리를 한다
START_KST = '2026-11-18 16:00'   # 시간대 시작(한국 시각, 포함)
END_KST = '2026-11-19 07:30'     # 시간대 끝(한국 시각, 미포함)
TIME_COLUMN = 'published_utc'    # 'published_utc'(발행 시각) 또는 'first_seen_utc'(처음 수집한 시각)
TOP_N = 3
MAX_ARTICLES_PER_ISSUE = 3
# ==============================

LLM_SOURCE_PATH = RESULT_DIR / 'llm_source.jsonl'   # 읽기만 한다
IMPORTANCE_PATH = RESULT_DIR / 'importance.csv'     # news_track의 매체 수(사건 매체 수 story_outlets를 읽기만 한다)
HOT_ISSUES_PATH = RESULT_DIR / 'macro_hot_issues.jsonl'   # 정리할 때마다 한 줄: 시간대, 고른 사건, 3줄
USAGE_PATH = RESULT_DIR / 'llm_usage.csv'           # LLM 호출마다 토큰 수(news_extract·news_track과 함께 씀)
KST = dt.timezone(dt.timedelta(hours=9))
assert ROOT.name == 'news_search', 'news_search 폴더에서 열어야 한다.'

## 2. 작은 도구

`news_extract`와 같은 도구다.
- `env_value`: proj 폴더 `.env`에서 키를 읽는다(값은 출력하지 않는다).
- `ask_openai`: OpenAI Responses API. 오류가 나면 오류 종류·코드만 알린다.

In [ ]:
def env_value(name):
    env_file = PROJ / '.env'
    if env_file.exists():
        for line in env_file.read_text(encoding='utf-8-sig').splitlines():
            key, _, value = line.partition('=')
            if key.strip() == name and value.strip():
                return value.strip().strip('"\'')
    return None


def read_jsonl(path):
    # 한 줄씩 읽는다. 다른 노트북이 쓰는 중인 마지막 줄(덜 쓰인 줄)은 건너뛴다.
    rows = []
    if path.exists():
        for line in path.read_text(encoding='utf-8').splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def append_jsonl(path, rows):
    with path.open('a', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


def append_csv(rows, path):
    if rows:
        data = pd.DataFrame(rows)
        if path.exists():   # 이미 있는 파일의 열 순서에 맞춘다
            data = data.reindex(columns=pd.read_csv(path, nrows=0, encoding='utf-8-sig').columns)
        data.to_csv(path, mode='a', header=not path.exists(), index=False, encoding='utf-8-sig')


class LLMError(Exception):
    pass


def ask_openai(instructions, user_message, text_format):
    request = urllib.request.Request(
        'https://api.openai.com/v1/responses', method='POST',
        headers={'Authorization': f"Bearer {env_value('OPENAI_API_KEY')}", 'Content-Type': 'application/json'},
        data=json.dumps({'model': LLM_MODEL, 'instructions': instructions, 'input': user_message, 'store': False,
                         'reasoning': {'effort': 'medium'}, 'text': {'format': text_format}}).encode('utf-8'))
    try:
        with urllib.request.urlopen(request, timeout=600) as response:
            body = json.loads(response.read().decode('utf-8'))
    except urllib.error.HTTPError as e:
        try:
            err = json.loads(e.read().decode('utf-8', 'replace')).get('error', {})
        except ValueError:
            err = {}
        raise LLMError(f"OpenAI HTTP {e.code} type={err.get('type')} code={err.get('code')}") from None
    text = ''.join(c.get('text', '') for item in body.get('output', []) if item.get('type') == 'message'
                   for c in item.get('content', []))
    return text, body.get('usage') or {}


def log_usage(step, ref, usage):
    append_csv([{'time_utc': dt.datetime.now(dt.timezone.utc).isoformat(timespec='seconds'), 'step': step, 'ref': ref,
                 'input_tokens': usage.get('input_tokens'),
                 'cached_tokens': (usage.get('input_tokens_details') or {}).get('cached_tokens'),
                 'output_tokens': usage.get('output_tokens')}], USAGE_PATH)


print('OPENAI_API_KEY:', '설정됨' if env_value('OPENAI_API_KEY') else '미설정')

## 3. 사건 고르기

1. `llm_source.jsonl`에서 `TIME_COLUMN`이 시간대 안인 기사만 남긴다.
2. `article_type`이 비어 있지 않은 기사(거시 유형이 하나라도 있는 기사)만 남긴다.
3. `story_id`가 있는 기사(사건 기사)만 남긴다. 분석·추천 글처럼 사건으로 묶이지 않은 기사는 `news_outlets`가 없어 뺀다.
4. 사건마다 한 줄: 그 사건의 기사 중 `news_outlets`가 가장 큰 기사(같으면 최근 발행)를 대표로 한다.
5. 사건 순위: `news_outlets` 큰 순 → 같으면 `story_outlets`(사건 전체 매체 수, `importance.csv`) 큰 순 → 같으면 대표 기사 발행 시각 최근 순. 위에서 `TOP_N`개.
6. LLM 입력: 사건마다 대표 기사부터 `MAX_ARTICLES_PER_ISSUE`건(시간대 안 거시 기사)의 제목·매체·발행 시각·short_summary·comparison_summary.

In [ ]:
def to_utc(text):
    t = pd.Timestamp(text)
    return (t.tz_localize(KST) if t.tzinfo is None else t).tz_convert('UTC')


def to_kst_text(utc_text):
    """UTC ISO 시각 → 한국 시각 ISO 문자열(예: 2026-11-16T05:42:00+09:00). 못 읽으면 빈 값."""
    t = pd.to_datetime(utc_text, utc=True, errors='coerce')
    return '' if pd.isna(t) else t.tz_convert(KST).isoformat(timespec='seconds')


def pick_issues(start_kst=None, end_kst=None, top_n=None):
    """시간대 안 거시·사건 기사(macro)와 고른 사건의 대표 기사(top)를 돌려준다."""
    start_kst, end_kst, top_n = start_kst or START_KST, end_kst or END_KST, top_n or TOP_N
    rows = pd.DataFrame(read_jsonl(LLM_SOURCE_PATH))
    if rows.empty:
        return rows, rows
    for c in ['article_type', 'story_id', 'news_outlets']:
        if c not in rows:
            rows[c] = None
    if rows['story_id'].isna().all():
        print('주의: story_id가 비어 있다 — news_track을 먼저 돌려야 한다.')
    rows['time'] = pd.to_datetime(rows[TIME_COLUMN], utc=True, format='ISO8601', errors='coerce')
    in_window = rows['time'].ge(to_utc(start_kst)) & rows['time'].lt(to_utc(end_kst))
    is_macro = rows['article_type'].map(lambda x: isinstance(x, list) and len(x) > 0)
    has_story = rows['story_id'].fillna('').astype(str) != ''
    macro = rows[in_window & is_macro & has_story].copy()
    macro['news_outlets'] = pd.to_numeric(macro['news_outlets'], errors='coerce').fillna(0).astype(int)
    macro = macro.sort_values(['news_outlets', 'time'], ascending=[False, False]).reset_index(drop=True)
    top = macro.drop_duplicates('story_id')
    if IMPORTANCE_PATH.exists():
        imp = pd.read_csv(IMPORTANCE_PATH, dtype=str).drop_duplicates('story_id').set_index('story_id')
        top['story_outlets'] = pd.to_numeric(top['story_id'].map(imp['story_outlets']), errors='coerce').fillna(0).astype(int)
    else:
        print('주의: importance.csv가 없다 — story_outlets를 0으로 두고 발행 시각으로만 가른다.')
        top['story_outlets'] = 0
    top = (top.sort_values(['news_outlets', 'story_outlets', 'time'], ascending=[False, False, False])
           .head(top_n).reset_index(drop=True))
    print(f'시간대 {start_kst} ~ {end_kst} (KST, {TIME_COLUMN}) | 시간대 안 기사 {int(in_window.sum())}건, '
          f'거시 {int((in_window & is_macro).sum())}건, 그중 사건 기사 {len(macro)}건 / 사건 {macro["story_id"].nunique()}개 → 상위 {len(top)}개')
    return macro, top


def text_or(value, default=''):
    return value if isinstance(value, str) and value else default


def issue_input(macro, top):
    """LLM에 넘길 이슈 목록(순위 순)."""
    issues = []
    for rank, (_, r) in enumerate(top.iterrows(), 1):
        same = macro[macro['story_id'] == r['story_id']].head(MAX_ARTICLES_PER_ISSUE)   # 대표 기사가 맨 앞
        issues.append({'rank': rank, 'article_type': r['article_type'], 'news_outlets': int(r['news_outlets']),
                       'articles': [{'title': a['title'], 'outlet': text_or(a.get('provider'), a['outlet']),
                                     'published_utc': a['published_utc'], 'short_summary': text_or(a.get('short_summary')),
                                     'comparison_summary': text_or(a.get('comparison_summary'))}
                                    for _, a in same.iterrows()]})
    return issues

## 4. 3줄 정리

- 고른 사건을 순위 순으로 OpenAI에 넘겨 `{"lines": [...]}`(사건마다 한 줄)을 받는다.
- 받은 줄을 `- ` 목록으로 보여 주고 `macro_hot_issues.jsonl`에 한 줄 남긴다(브리핑 날짜 `date`(시간대 끝의 한국 날짜), 생성 시각 `created_kst`(한국), 시간대, 고른 사건 번호·대표 기사 key·대표 기사 발행 시각(한국, `published_kst`)·매체 수, 3줄).
- 고른 사건이 3개보다 적으면 그 수만큼만 쓴다. 0개면 OpenAI를 부르지 않는다.

In [ ]:
HOT_PROMPT = HOT_PROMPT_PATH.read_text(encoding='utf-8')


def hot_issues(start_kst=None, end_kst=None):
    start_kst, end_kst = start_kst or START_KST, end_kst or END_KST
    macro, top = pick_issues(start_kst, end_kst)
    if top.empty:
        print('정리할 거시 이슈가 없다.')
        return []
    display(top[['story_id', 'news_outlets', 'story_outlets', 'published_utc', 'article_type', 'title']])
    issues = issue_input(macro, top)
    if not RUN_LLM:
        print('RUN_LLM = False라 OpenAI를 부르지 않는다. LLM에 넘길 입력:')
        print(json.dumps({'issues': issues}, ensure_ascii=False, indent=1))
        return []
    text, usage = ask_openai(HOT_PROMPT, 'issues json:\n' + json.dumps({'issues': issues}, ensure_ascii=False), {'type': 'json_object'})
    log_usage('hot_issue', f'{start_kst}~{end_kst}', usage)
    lines = [str(x).strip().lstrip('-•· ').strip() for x in json.loads(text).get('lines', [])]
    if len(lines) != len(issues):
        print(f'주의: 받은 줄 {len(lines)}개, 이슈 {len(issues)}개')
    now = dt.datetime.now(dt.timezone.utc)
    append_jsonl(HOT_ISSUES_PATH, [{
        'date': to_utc(end_kst).tz_convert(KST).strftime('%Y-%m-%d'),   # 브리핑 날짜(시간대 끝의 한국 날짜)
        'created_kst': now.astimezone(KST).strftime('%Y-%m-%d %H:%M:%S'),   # 생성 시각(한국)
        'start_kst': start_kst, 'end_kst': end_kst, 'time_column': TIME_COLUMN, 'model': LLM_MODEL,
        'story_ids': top['story_id'].tolist(), 'keys': top['key'].tolist(),
        'published_kst': [to_kst_text(x) for x in top['published_utc']],   # 대표 기사 발행 시각(한국)
        'news_outlets': top['news_outlets'].tolist(), 'story_outlets': top['story_outlets'].tolist(),
        'lines': lines}])
    print('\n'.join('- ' + l for l in lines))
    return lines

## 5. 실행

In [ ]:
lines = hot_issues()